# Alert escalation — reproducible pipeline (team FB203632)

This notebook rebuilds the submitted predictions from the four raw files in `data/`:

1. build per-alert features from each alert's own transactions (no statistic is ever computed on test data);
2. recreate the fixed folds (stratified 5-fold × 3 repeats, seeds 42/43/44);
3. re-run every ensemble member with the configuration accepted in `EXPERIMENTS_LOG.md`;
4. combine the accepted members by rank average (the final configuration is a single LightGBM: once the cell-interaction features were added, no other model passed the rule), calibrate with Platt scaling fitted on training out-of-fold predictions, write `submission/team_FB203632.csv`.

Every choice below was made by an experiment with the acceptance rule of `PLAN.md` §4 (mean OOF AUC +0.0010 and a gain in all three repeats). The experiment history is in `EXPERIMENTS_LOG.md`; failures and their reasons are in `ANTI_PATTERNS.md`.

In [1]:
import sys, json, time, platform
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, lightgbm, xgboost, catboost, sklearn
from src.store import FeatureStore
from src.cv import make_folds, get_folds, FOLDS_PATH
from src.final import run_member, ensemble_and_submit
from src.features import BASE_FAMILIES, FOLD_FAMILIES
from src.config import REPEAT_SEEDS
print('python', platform.python_version(), '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| catboost', catboost.__version__, '| sklearn', sklearn.__version__)

python 3.12.4 | lightgbm 4.5.0 | xgboost 2.1.3 | catboost 1.2.7 | sklearn 1.5.2


## 1. Accepted configuration

Copied from the experiment state when this notebook was generated. `families` are feature-family prefixes (see `src/features.py`), `spec` the model and its tuned parameters.

In [2]:
CONFIG = json.loads(r"""{
 "members": {
  "lgb": {
   "families": [
    "f1",
    "f7",
    "fi",
    "fx"
   ],
   "spec": {
    "type": "lgb",
    "params": {},
    "n_estimators": 140
   },
   "model_seeds": 1,
   "selection": null,
   "txm": false,
   "logged_result": "LOO_f1+f7+fi+fx"
  }
 },
 "ensemble": {
  "members": [
   "lgb"
  ],
  "kind": "equal",
  "weights": [
   1.0
  ]
 }
}""")
print(json.dumps(CONFIG, indent=1))

{
 "members": {
  "lgb": {
   "families": [
    "f1",
    "f7",
    "fi",
    "fx"
   ],
   "spec": {
    "type": "lgb",
    "params": {},
    "n_estimators": 140
   },
   "model_seeds": 1,
   "selection": null,
   "txm": false,
   "logged_result": "LOO_f1+f7+fi+fx"
  }
 },
 "ensemble": {
  "members": [
   "lgb"
  ],
  "kind": "equal",
  "weights": [
   1.0
  ]
 }
}


## 2. Features from raw data

`FeatureStore(rebuild=True)` re-reads the raw files and recomputes everything. The final model uses `f1` (amount statistics per type × direction cell), `f7` (recent-vs-history contrasts), `fi` (differences of amount levels between cells) and `fx` (means of exp(σ·amount)); all four use only each alert's own transactions. The other families in `src/features.py` and `src/features_extra.py` were tested and rejected (`EXPERIMENTS_LOG.md`); fold-dependent ones are fitted on training folds only. No transaction or alert is dropped (asserted inside `src/data.py`).

In [3]:
t0 = time.time()
fs = FeatureStore(rebuild=True)
print('alerts train/test:', len(fs.y), len(fs.sig['test']), '| transactions train/test:', len(fs.tx['train']), len(fs.tx['test']))
assert list(fs.base['train'].columns) == list(fs.base['test'].columns)
print(pd.Series([c.split('_', 1)[0] for c in fs.base['train'].columns]).value_counts().to_dict())
print(f'{time.time() - t0:.0f}s')

FeatureStore ready in 31.2s; base features: 293
alerts train/test: 14000 6000 | transactions train/test: 6987663 3027575
{'f1': 164, 'f5': 50, 'f4': 24, 'f3': 15, 'fx': 14, 'f7': 12, 'f2': 11, 'f6': 3}
31s


## 3. Fixed folds

Stratified 5-fold, repeats with seeds 42, 43, 44 (scoring) and 100 (used only for tuning). Recomputed here and compared with the folds used in the experiments.

In [4]:
folds = make_folds(fs.y)
if FOLDS_PATH.exists():
    stored = get_folds(fs.y)
    assert all((stored[s] == folds[s]).all() for s in folds), 'folds differ from the experiments'
    print('folds identical to the experiment folds')
else:
    get_folds(fs.y)
pd.DataFrame({s: pd.Series(fs.y).groupby(folds[s]).agg(['size', 'mean']).round(4).astype(str).agg(' / '.join, axis=1) for s in REPEAT_SEEDS})

folds identical to the experiment folds


,42,43,44
0,2800 / 0.1718,2800 / 0.1718,2800 / 0.1718
1,2800 / 0.1718,2800 / 0.1718,2800 / 0.1718
2,2800 / 0.1718,2800 / 0.1718,2800 / 0.1718
3,2800 / 0.1718,2800 / 0.1718,2800 / 0.1718
4,2800 / 0.1718,2800 / 0.1718,2800 / 0.1718


## 4. Re-run the ensemble members

Each member is trained on the 3 × 5 folds; out-of-fold predictions give the CV score, and every fold model predicts the test alerts. The table compares the AUC with the value logged during the experiments.

In [5]:
from src.cv import load_result
txf = None
if any(m['txm'] for m in CONFIG['members'].values()):
    from src.txmodel import TxScoreFeatures
    txf = TxScoreFeatures(fs, cache_name='txm_cache_notebook.pkl', use_cache=False)
results, rows = {}, []
for name, member in CONFIG['members'].items():
    t0 = time.time()
    results[name] = run_member(fs, member, name='NB_' + name, txf=txf)
    try:
        logged = load_result(member['logged_result'])['auc']
    except FileNotFoundError:
        logged = {s: np.nan for s in REPEAT_SEEDS}
    for s in REPEAT_SEEDS:
        rows.append({'member': name, 'repeat': s, 'auc_notebook': round(results[name]['auc'][s], 5), 'auc_logged': round(logged[s], 5)})
    print(name, f'{time.time() - t0:.0f}s')
pd.DataFrame(rows)

  [NB_lgb] seed 42: AUC 0.65269  (2s)


  [NB_lgb] seed 43: AUC 0.65388  (3s)


  [NB_lgb] seed 44: AUC 0.65173  (5s)


lgb 5s


,member,repeat,auc_notebook,auc_logged
0,lgb,42,0.65269,0.65269
1,lgb,43,0.65388,0.65388
2,lgb,44,0.65173,0.65173


## 5. Ensemble, calibration, submission

Members are rank-averaged per repeat. Test predictions are converted to percentiles through the ECDF of each member's *training* out-of-fold predictions, and Platt scaling is fitted on training out-of-fold scores only, so nothing is fitted on the test set. Platt scaling is monotone and does not change the AUC.

In [6]:
out = ensemble_and_submit(fs, results, CONFIG['ensemble'])
print('final CV ROC-AUC per repeat:', {s: round(v, 5) for s, v in out['auc'].items()})
print(f"final CV ROC-AUC: {out['mean']:.4f} ± {out['std']:.4f}")
print('written:', out['path'])
out['sub'].head()

final CV ROC-AUC per repeat: {42: 0.65269, 43: 0.65388, 44: 0.65173}
final CV ROC-AUC: 0.6528 ± 0.0009
written: C:\kaggle\WIUT_hackathon\submission\team_FB203632.csv


,signal_id,ehtimollik
0,SG_000001,0.113151
1,SG_000007,0.209436
2,SG_000009,0.124072
3,SG_000010,0.113715
4,SG_000011,0.156253


In [7]:
sub = pd.read_csv(out['path'])
assert list(sub.columns) == ['signal_id', 'ehtimollik']
assert len(sub) == len(fs.sig['test']) and sub.signal_id.is_unique
assert sub.ehtimollik.between(0, 1).all() and sub.ehtimollik.notna().all()
print('rows', len(sub), '| mean predicted probability %.4f (train escalation rate %.4f)' % (sub.ehtimollik.mean(), fs.y.mean()))
sub.ehtimollik.describe()

rows 6000 | mean predicted probability 0.1730 (train escalation rate 0.1718)


count    6000.000000
mean        0.173036
std         0.072983
min         0.024325
25%         0.120574
50%         0.159527
75%         0.212945
max         0.568738
Name: ehtimollik, dtype: float64

## 6. Where the decisions come from

- `PLAN.md` — validation scheme and acceptance rule (fixed before modelling).
- `EXPERIMENTS_LOG.md` — one line per experiment with Δ AUC per repeat and the decision.
- `ANTI_PATTERNS.md` — what did not work and why.
- `experiments/` — the scripts that ran every experiment.
- EDA website: https://screnn7.github.io/WIUT_hackathon/